# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 3 — Hive: tabelas Managed vs External

## 🎯 Objetivo

Criar tabelas Hive sobre os dados do HDFS e comprovar na prática a diferença entre Managed e External — inclusive o que acontece com `DROP TABLE`.

**Importante:** iremos reproduzir a ideia central do Lab 3 usando **DuckDB + Python**, sem precisar de um cluster Hadoop/Hive real.

> ⚠️ Esta versão segue a **Rota B** do laboratório. DuckDB não reproduz exatamente o comportamento de uma Managed Table do Hive; o experimento demonstra conceitualmente que o `DROP` da tabela não apaga o CSV fonte.


## Configuração inicial

In [7]:
# Instala a biblioteca DuckDB
!pip -q install duckdb


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\aj_ol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
# Permite o upload de arquivos para o Colab
# from google.colab import files
# uploaded = files.upload()
# print("Arquivos enviados:", list(uploaded.keys()))

In [9]:
# Prepara o ambiente criando diretórios e copiando o CSV de clientes
from pathlib import Path
import shutil

BASE = Path("../content/bigdata/")
RAW_CUSTOMERS = BASE / "raw" / "customers"
RAW_TRANSACTIONS = BASE / "raw" / "transactions"
RAW_FRAUD = BASE / "raw" / "fraud_labels"

for p in [RAW_CUSTOMERS, RAW_TRANSACTIONS, RAW_FRAUD,
          BASE / "bronze", BASE / "silver", BASE / "gold"]:
    p.mkdir(parents=True, exist_ok=True)

customer_file = Path("../customers_synthetic.csv")
if not customer_file.exists():
    raise FileNotFoundError(
        "Não encontrei customers_synthetic.csv. Envie o arquivo e execute esta célula novamente."
    )

target_customer = RAW_CUSTOMERS / "customers_synthetic.csv"
shutil.copy2(customer_file, target_customer)
print("CSV disponível em:", target_customer)

CSV disponível em: ..\content\bigdata\raw\customers\customers_synthetic.csv


## Passo 1 — Abrir o Python com DuckDB

In [10]:
# Conecta ao banco de dados DuckDB
import duckdb
import os

con = duckdb.connect("/content/bigdata_course.duckdb")
print("DuckDB conectado com sucesso.")

DuckDB conectado com sucesso.


## Passo 2 — "Criar tabela" apontando para o CSV (equivalente a External)

No Hive, uma tabela External aponta para arquivos existentes no HDFS. Aqui fazemos a mesma ideia conceitualmente: o DuckDB lê diretamente o CSV, sem copiar seus dados para dentro da tabela.


In [11]:
# Cria a VIEW 'raw_customers' a partir do arquivo CSV (simula tabela External)
csv_path = str(target_customer)

con.execute("DROP VIEW IF EXISTS raw_customers")
con.execute(f'''
CREATE VIEW raw_customers AS
SELECT *
FROM read_csv_auto('{csv_path}', header=true)
''')

print("VIEW raw_customers criada.")

VIEW raw_customers criada.


In [12]:
# Conta e verifica o número de registros em 'raw_customers'
count_customers = con.execute(
    "SELECT COUNT(*) FROM raw_customers"
).fetchone()[0]

print("Quantidade de clientes:", count_customers)
assert count_customers == 9993, f"Esperado: 9993 | Encontrado: {count_customers}"

Quantidade de clientes: 9993


In [13]:
# Exibe as 5 primeiras linhas de 'raw_customers'
con.execute("""
SELECT *
FROM raw_customers
LIMIT 5
""").df()

,customer_id,name,cpf,email,segment,credit_score,created_at
0,1,Ana Laura Campos,943.065.218-42,igor46@example.com,Premium,426,2026-07-03
1,2,Mariah Caldeira,586.237.094-38,jose48@example.com,High-Risk,481,2026-06-17
2,3,Kevin Cavalcante,530.629.814-15,theoda-costa@example.org,Standard,708,2025-11-06
3,4,Maria Laura Freitas,725.130.864-90,castrolucas@example.org,Premium,473,2026-02-07
4,5,Srta. Mirella Moura,570.814.239-14,emilly20@example.com,Premium,816,2025-10-04


## Passo 3 — Demonstrar Managed vs External com um teste

No Hive real, uma Managed Table pertence ao Hive. No laboratório, o `DROP TABLE` pode remover os dados associados.

No DuckDB, vamos criar uma tabela física de teste com alguns registros. Depois do `DROP`, verificaremos se o CSV original continua intacto.


In [14]:
# Cria uma tabela física 'teste_managed' com 5 registros de 'raw_customers'
con.execute("DROP TABLE IF EXISTS teste_managed")
con.execute("""
CREATE TABLE teste_managed AS
SELECT *
FROM raw_customers
LIMIT 5
""")

print("Conteúdo da tabela teste_managed:")
con.execute("SELECT * FROM teste_managed").df()

Conteúdo da tabela teste_managed:


,customer_id,name,cpf,email,segment,credit_score,created_at
0,1,Ana Laura Campos,943.065.218-42,igor46@example.com,Premium,426,2026-07-03
1,2,Mariah Caldeira,586.237.094-38,jose48@example.com,High-Risk,481,2026-06-17
2,3,Kevin Cavalcante,530.629.814-15,theoda-costa@example.org,Standard,708,2025-11-06
3,4,Maria Laura Freitas,725.130.864-90,castrolucas@example.org,Premium,473,2026-02-07
4,5,Srta. Mirella Moura,570.814.239-14,emilly20@example.com,Premium,816,2025-10-04


In [15]:
# Conta e verifica o número de linhas em 'teste_managed'
managed_count = con.execute(
    "SELECT COUNT(*) FROM teste_managed"
).fetchone()[0]

print("Linhas em teste_managed:", managed_count)
assert managed_count == 5

Linhas em teste_managed: 5


In [16]:
# Remove a tabela 'teste_managed' e verifica se o arquivo CSV original permanece
con.execute("DROP TABLE teste_managed")

arquivo_existe = os.path.exists(csv_path)
print("Tabela teste_managed removida.")
print("Arquivo original ainda existe?", arquivo_existe)

assert arquivo_existe is True

Tabela teste_managed removida.
Arquivo original ainda existe? True


## Passo 4 — O que isso ensina

Resultado esperado:

```text
Arquivo original ainda existe? True
```

- **DuckDB / Rota B:** o CSV fonte permanece após `DROP TABLE`.
- **Hive Managed / Rota A:** o Hive é dono dos dados e o `DROP TABLE` remove os dados associados.
- **Hive External / Rota A:** o `DROP TABLE` remove o metadado, mas preserva os arquivos externos.

Portanto, esta simulação demonstra o comportamento de **External**, mas não reproduz integralmente a semântica de `DROP` de uma Managed Table do Hive.


## Comparação Managed vs External

| Característica | Hive Managed | Hive External | Simulação DuckDB |
|---|---|---|---|
| Tabela registrada no catálogo | Sim | Sim | Sim |
| Dados associados à tabela | Sim | Não necessariamente | Não |
| `DROP TABLE` apaga o arquivo fonte | **Sim** | **Não** | **Não** |
| Arquivo externo continua disponível | Não | Sim | Sim |


In [17]:
# Verifica se a VIEW 'raw_customers' e o arquivo CSV original ainda estão intactos
final_count = con.execute(
    "SELECT COUNT(*) FROM raw_customers"
).fetchone()[0]

print("Clientes disponíveis após o experimento:", final_count)
print("CSV fonte ainda existe?", os.path.exists(csv_path))

assert final_count == 9993
assert os.path.exists(csv_path)

Clientes disponíveis após o experimento: 9993
CSV fonte ainda existe? True


## ✅ Checkpoint

- [x] `raw_customers` criada sobre o CSV.
- [x] `raw_customers` possui 9.993 linhas.
- [x] Uma tabela física `teste_managed` foi criada.
- [x] `DROP TABLE teste_managed` foi executado.
- [x] O CSV original permaneceu intacto.
- [x] A diferença entre Managed e External foi demonstrada conceitualmente.
- [x] `raw_customers` continua disponível para o próximo lab.